# RLHF: 인간 피드백 기반 강화학습 - 실습 코드 1: RLHF Reward Model 학습 (TRL)

- Tutorial ID: `expand-rlhf`
- Tutorial: RLHF: 인간 피드백 기반 강화학습
- Section ID: `expand-rlhf-code-1`
- Section: 실습 코드 1: RLHF Reward Model 학습 (TRL)


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: RLHF Reward Model 학습 (TRL)
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) 선호쌍 chosen/rejected가 loss와 policy update 신호로 바뀌는 흐름 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from trl import RewardTrainer, RewardConfig
from datasets import Dataset

# 보상 모델 초기화
model = AutoModelForSequenceClassification.from_pretrained(
    "meta-llama/Llama-2-7b-hf", num_labels=1
)
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-hf")

# 선호도 데이터 (chosen > rejected)
data = [
    {"prompt": "Explain AI", 
     "chosen": "AI is the simulation of human intelligence...",
     "rejected": "AI is robots and stuff"},
    # ... 더 많은 데이터
]
dataset = Dataset.from_list(data)

# 보상 모델 학습
training_args = RewardConfig(
    output_dir="./reward_model",
    per_device_train_batch_size=4,
    learning_rate=1e-5,
    num_train_epochs=1,
)
trainer = RewardTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=dataset,
)
trainer.train()